## Train pi0.5

In [ ]:
from datetime import datetime
import os
os.environ["HF_TOKEN"]="insert_huggingface_token_here"
os.environ["WANDB_API_KEY"]="insert_wandb_api_key_here"
## Set NCCL environment variables for distributed training in GCP G4 
## single instance, multi-GPU setup. Adjust the interface name (e.g., "ens3") as needed for your specific instance type.
os.environ.pop("NCCL_NET", None) # on single instance, multi-GPU setup, NCCL_NET should not be set to "IB" or "TCP"
os.environ["NCCL_SOCKET_IFNAME"] = "ens3" # Adjust "ens3" to the correct network interface for your GCP instance (e.g., "ens4", "eth0", etc.)
os.environ["NCCL_P2P_LEVEL"] = "PHB" # Set the P2P level to "PHB" (PCIe Host Bridge) for optimal GPU communication on a single instance
os.environ["TOKENIZERS_PARALLELISM"] = "false" # Disable parallelism in tokenizers to avoid potential issues with multiprocessing
import warnings
warnings.filterwarnings("ignore")


In [ ]:
!rm -rf ckpt #remove checkpoint directory if it already exists to avoid conflicts with previous runs
!rm -rf logs #remove logs directory if it already exists to avoid conflicts with previous runs

In [ ]:

DATASET_REPO="gimarchetti/ur5-experiment-dataset" #@param {type:"string"}
DATASET_ROOT="./dataset/teleoperation_dataset" #@param {type:"string"}
POLICY_REPO="gimarchetti/ur5-experiment-pi05" #@param {type:"string"}
OUTPUT_DIR="./ckpt/ur5-experiment-pi05" #@param {type:"string"}
JOB_NAME="ur5-experiment-pi05"+datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
MAX_TRAIN_STEPS=400 #@param {type:"integer"}
CHUNK_SIZE=50 #@param {type:"integer"}
ACTION_STEPS=10 #@param {type:"integer"}
#EVAL_STEPS=1000
#SAVE_STEPS=1000
BATCH_SIZE=64 #@param {type:"integer"}
#LEARNING_RATE=5e-5
#WEIGHT_DECAY=0.01
#WARMUP_STEPS=500
#LOGGING_STEPS=100


In [ ]:
#--dataset.root={DATASET_ROOT} \
!hf download {DATASET_REPO} --repo-type dataset --local-dir {DATASET_ROOT} 

In [ ]:
# Train the policy pi05 with the specified configuration. The model will be pushed to the Hugging Face Hub under the provided POLICY_REPO name after training.
!accelerate launch \
--multi_gpu \
--num_machines=1 \
--num_processes=4 \
--mixed_precision=bf16 \
$(which lerobot-train) \
    --dataset.repo_id={DATASET_REPO} \
    --dataset.root={DATASET_ROOT} \
    --policy.type=pi05 \
    --policy.push_to_hub=true \
    --policy.repo_id={POLICY_REPO} \
    --output_dir={OUTPUT_DIR} \
    --job_name={JOB_NAME} \
    --policy.pretrained_path=lerobot/pi05_base \
    --policy.compile_model=false \
    --policy.gradient_checkpointing=true \
    --wandb.enable=true \
    --policy.dtype=bfloat16 \
    --policy.freeze_vision_encoder=false \
    --policy.train_expert_only=false \
    --steps={MAX_TRAIN_STEPS} \
    --log_freq=50 \
    --eval_freq=-1 \
    --policy.device=cuda \
    --policy.chunk_size={CHUNK_SIZE} \
    --policy.n_action_steps={ACTION_STEPS} \
    --batch_size={BATCH_SIZE}


## Train GR00T N 1.5 TO BE DONE YET

In [ ]:
!pip install ninja "packaging>=24.2,<26.0"
!pip install peft
!pip install dm-tree==0.1.9
!pip install -U transformers
!pip install flash-attn==2.7.3 --no-build-isolation

In [ ]:
!lerobot-train\
    --dataset.repo_id=Jeongeun/tutorial_v2 \
    --dataset.root=dataset/leader_data \
    --policy.type=groot \
    --policy.repo_id=={YOUR REPO} \
    --output_dir=ckpt/tutorial_v2_groot \
    --job_name=tutorial_v2_groot \
    --wandb.enable=false \
    --steps=20000 \
    --policy.chunk_size=20 \
    --policy.n_action_steps=20 \
    --batch_size=32